# [16.3] Shapley Interactions with shapiq - Solutions

This notebook runs the reference implementation, visible tests, and direct checks on the committed CUDA verification report.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter16_shapley_attribution_baselines"
section = "part3_shapley_interactions_shapiq"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_shapley_interactions_shapiq.solutions as solutions
import part3_shapley_interactions_shapiq.tests as tests

In [ ]:
tests.test_additive_game_enumerates_complete_zero_interaction_table(
    solutions.additive_game,
    solutions.pairwise_shapley_interactions,
)
tests.test_interaction_game_recovers_target_pair_delta(
    solutions.interaction_game,
    solutions.pairwise_shapley_interactions,
)
tests.test_pairwise_interaction_report_matches_reference_and_rejects_spurious_pairs(
    solutions.pairwise_interaction_report,
)
tests.test_shapiq_interaction_parity_report_matches_exact_sii(
    solutions.shapiq_interaction_parity_report,
)
tests.test_neural_game_value_table_contains_planted_interactions(
    solutions.binary_feature_table,
    solutions.true_neural_game_scores,
    solutions.coalition_table_from_true_game,
    solutions.pairwise_shapley_interactions,
)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
smoke = solutions.run_smoke_test(cpu=True)
assert smoke["additive_interactions"]["max_abs_interaction"] == 0.0
assert smoke["target_pair"]["recovers_interaction"]
assert abs(smoke["target_pair"]["target_interaction"] - 1.0) < 1e-9
assert smoke["shapiq_parity"]["shapiq_available"]
assert smoke["shapiq_parity"]["matches_shapiq"]
smoke

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

assert report["accepted"] and report["tests_passed"]
assert report["gt_tier"] == "GT-0"
assert report["notebook_id"] == "16_3_shapley_interactions_with_shapiq"
assert gpu["cuda_available"] and gpu["preflight_passed"]
assert gpu["device"] == "NVIDIA GeForce RTX 5090 Laptop GPU"
assert gpu["model_family"] == "cuda_trained_neural_coalition_game_mlp"
assert gpu["num_players"] == 4
assert gpu["coalition_count"] == 16
assert gpu["training_example_count"] == 16
assert gpu["training_steps"] == 1200
assert gpu["fit_mse"] <= 1e-8
assert gpu["fit_max_abs_error"] <= 1e-4
assert gpu["positive_interaction_pair"] == [0, 2]
assert gpu["negative_interaction_pair"] == [1, 3]
assert gpu["positive_interaction_value"] > 0
assert gpu["negative_interaction_value"] < 0
assert gpu["interaction_max_abs_error"] <= 1e-4
assert gpu["max_spurious_interaction"] <= 1e-4
assert gpu["interaction_signs_recovered"]
assert gpu["shapiq_available"] and gpu["shapiq_matches"]
assert gpu["shapiq_max_abs_error"] <= 1e-5
assert gpu["shuffled_control_interaction_error"] >= 1.0
assert gpu["shuffled_control_rejected"]
assert gpu["peak_vram_gb"] < 1.0
assert gpu["within_vram_budget"]

tests.test_committed_gpu_report_matches_shapley_interaction_contract(gpu)
{
    "device": gpu["device"],
    "positive_interaction_value": gpu["positive_interaction_value"],
    "negative_interaction_value": gpu["negative_interaction_value"],
    "shapiq_max_abs_error": gpu["shapiq_max_abs_error"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}